# 第3章：基于直方图统计的处理

## 编程实践：直方图、均衡化与匹配

---

## 一、直方图基础

### 1.1 什么是图像直方图？

图像直方图是图像中**像素值分布**的统计图。它显示每个灰度值在图像中出现的频率。

```
        像素数
        30┤    ██
        25┤    ██
        20┤  ██ ██
        15┤  ██ ██
        10┤  ██ ██ ██
         5┤█ ██ ██ ██
         0┼──────────── 灰度值
          0  64  128 192 255
```

### 1.2 直方图的作用
- **描述图像亮度分布**：整体偏暗/偏亮/对比度低
- **图像增强**：通过修改直方图改善视觉效果
- **图像匹配**：比较两张图的灰度分布相似性
- **阈值选择**：帮助确定二值化的阈值

### 1.3 直方图均衡化

直方图均衡化将图像的直方图**拉伸为均匀分布**，使图像的灰度值在整个范围内均匀分布。

```
变换前 (集中在暗部):  变换后 (均匀分布):
    ████                 ██
    ██████               ██
    ████████             ██
    ██████████           ██
    ████████████    →    ██
    ██████████████       ██
    ████████████████     ██
    ██████████████████   ██
```

### 1.4 直方图匹配

直方图匹配将图像的直方图**转换为参考图像的直方图**，使源图具有与参考图相似的灰度分布。


## 二、实现要求

### 实践1：计算并可视化灰度图像的像素直方图
> 读取灰度图像，统计像素分布直方图并可视化。除 OpenCV 读写外，其余代码手写。

### 实践2：直方图均衡化
> 读取灰度图像，计算直方图并进行均衡化，保存结果。除 OpenCV 读写外，其余代码手写。

### 实践3：直方图匹配
> 读取两张灰度图像（源图和参考图），分别计算直方图，进行直方图匹配，保存结果。除 OpenCV 读写外，其余代码手写。


In [ ]:
# 导入库
import cv2
import numpy as np

print(f"OpenCV 版本: {cv2.__version__}")

In [ ]:
# 生成本章所需的测试图像
import numpy as np

print("正在生成测试图像...")

# 1. 灰度测试图像
h, w = 400, 500
gray = np.zeros((h, w), dtype=np.uint8)
for y in range(h):
    for x in range(w):
        gray[y, x] = int(255 * x / w)
cv2.rectangle(gray, (50, 50), (150, 150), 50, -1)
cv2.rectangle(gray, (200, 50), (300, 150), 128, -1)
cv2.rectangle(gray, (350, 50), (450, 150), 200, -1)
cv2.imwrite("gray_image.jpg", gray)

# 2. 直方图源图像 (偏暗)
h2, w2 = 300, 400
src = np.zeros((h2, w2), dtype=np.uint8)
for y in range(h2):
    for x in range(w2):
        src[y, x] = int(30 + 50 * x / w2)
cv2.imwrite("hist_source.jpg", src)

# 3. 直方图参考图像 (偏亮)
ref = np.zeros((h2, w2), dtype=np.uint8)
for y in range(h2):
    for x in range(w2):
        ref[y, x] = int(150 + 70 * x / w2)
cv2.imwrite("hist_reference.jpg", ref)

print("所有测试图像已生成!")

In [ ]:
def compute_histogram_manual(image):
    """
    手写直方图计算 (单通道图像)
    统计每个灰度值(0-255)出现的次数
    """
    # 初始化直方图数组 (256 个灰度级)
    histogram = [0] * 256
    
    height, width = image.shape[:2]
    
    # 遍历每个像素, 统计每个灰度值的出现次数
    for y in range(height):
        for x in range(width):
            pixel_value = image[y, x]
            histogram[pixel_value] += 1
    
    return histogram

In [ ]:
# ==================== 实践1: 计算直方图 ====================

# 读取灰度图像
gray_img = cv2.imread("gray_image.jpg", cv2.IMREAD_GRAYSCALE)

if gray_img is not None:
    h, w = gray_img.shape
    print(f"读取灰度图像成功! 尺寸: {w}x{h}")
    
    # 计算直方图
    print("正在计算直方图...")
    hist = compute_histogram_manual(gray_img)
    
    # 打印直方图统计信息
    total_pixels = sum(hist)
    print(f"总像素数: {total_pixels}")
    
    # 找到直方图的峰值
    peak_value = max(hist)  # 最大频数
    peak_bin = hist.index(peak_value)  # 对应的灰度值
    print(f"直方图峰值: 灰度值={peak_bin}, 像素数={peak_value}")
    
    # 计算平均灰度值
    mean_val = sum(i * hist[i] for i in range(256)) / total_pixels
    print(f"平均灰度值: {mean_val:.2f}")
    
    # 可视化直方图
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 显示图像
    axes[0].imshow(gray_img, cmap='gray')
    axes[0].set_title('Gray Image')
    axes[0].axis('off')
    
    # 显示直方图 (柱状图)
    bins = list(range(256))
    axes[1].bar(bins, hist, width=1, color='steelblue', edgecolor='none')
    axes[1].set_title('Pixel Histogram')
    axes[1].set_xlabel('Gray Value (0-255)')
    axes[1].set_ylabel('Pixel Count')
    axes[1].set_xlim(0, 255)
    
    plt.tight_layout()
    plt.show()
else:
    print("读取图像失败!")

In [ ]:
def histogram_equalization_manual(image):
    """
    手写直方图均衡化
    步骤:
    1. 计算原图像直方图
    2. 计算累积分布函数 (CDF)
    3. 通过 CDF 映射像素值
    """
    # Step 1: 计算直方图
    hist = compute_histogram_manual(image)
    
    height, width = image.shape[:2]
    total_pixels = height * width
    
    # Step 2: 计算累积分布函数 (CDF)
    # CDF[i] = 直方图中 0~i 的累积像素数 / 总像素数
    cdf = [0.0] * 256
    cdf[0] = hist[0] / total_pixels
    for i in range(1, 256):
        cdf[i] = cdf[i-1] + hist[i] / total_pixels
    
    # Step 3: 构建映射表
    # 将 CDF 值映射回 [0, 255]
    lut = [int(cdf[i] * 255) for i in range(256)]
    
    # Step 4: 应用映射到每个像素
    result = image.copy()
    for y in range(height):
        for x in range(width):
            result[y, x] = lut[image[y, x]]
    
    return result

In [ ]:
# ==================== 实践2: 直方图均衡化 ====================

if gray_img is not None:
    print("正在进行直方图均衡化...")
    
    # 执行均衡化
    equalized = histogram_equalization_manual(gray_img)
    
    # 保存结果
    cv2.imwrite("equalized_image.jpg", equalized)
    print("均衡化完成! 已保存: equalized_image.jpg")
    
    # 计算均衡化后的直方图
    hist_eq = compute_histogram_manual(equalized)
    
    # 可视化对比
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 原图
    axes[0, 0].imshow(gray_img, cmap='gray')
    axes[0, 0].set_title('Original Image')
    axes[0, 0].axis('off')
    
    # 均衡化后
    axes[0, 1].imshow(equalized, cmap='gray')
    axes[0, 1].set_title('Equalized Image')
    axes[0, 1].axis('off')
    
    # 原直方图
    axes[1, 0].bar(range(256), hist, width=1, color='steelblue')
    axes[1, 0].set_title('Original Histogram')
    axes[1, 0].set_xlim(0, 255)
    
    # 均衡化后直方图
    axes[1, 1].bar(range(256), hist_eq, width=1, color='coral')
    axes[1, 1].set_title('Equalized Histogram')
    axes[1, 1].set_xlim(0, 255)
    
    plt.tight_layout()
    plt.show()
    
    # 统计对比
    print(f"\n原图 - 均值: {gray_img.mean():.1f}, 标准差: {gray_img.std():.1f}")
    print(f"均衡化后 - 均值: {equalized.mean():.1f}, 标准差: {equalized.std():.1f}")
    print("\n说明: 均衡化后直方图更均匀, 对比度提高")

In [ ]:
def histogram_matching_manual(source_img, reference_img):
    """
    手写直方图匹配
    将 source_img 的直方图匹配到 reference_img 的直方图
    """
    # Step 1: 计算源图直方图和 CDF
    source_hist = compute_histogram_manual(source_img)
    h_s, w_s = source_img.shape[:2]
    total_s = h_s * w_s
    
    # 源图 CDF
    source_cdf = [0.0] * 256
    source_cdf[0] = source_hist[0] / total_s
    for i in range(1, 256):
        source_cdf[i] = source_cdf[i-1] + source_hist[i] / total_s
    
    # Step 2: 计算参考图直方图和 CDF
    ref_hist = compute_histogram_manual(reference_img)
    h_r, w_r = reference_img.shape[:2]
    total_r = h_r * w_r
    
    # 参考图 CDF
    ref_cdf = [0.0] * 256
    ref_cdf[0] = ref_hist[0] / total_r
    for i in range(1, 256):
        ref_cdf[i] = ref_cdf[i-1] + ref_hist[i] / total_r
    
    # Step 3: 构建映射表
    # 对源图的每个灰度值, 找到参考图中 CDF 最接近的灰度值
    lut = [0] * 256
    for i in range(256):
        # 源图灰度值 i 的 CDF 值
        cdf_val = source_cdf[i]
        
        # 在参考图 CDF 中找最接近的值
        min_diff = float('inf')
        best_match = 0
        for j in range(256):
            diff = abs(ref_cdf[j] - cdf_val)
            if diff < min_diff:
                min_diff = diff
                best_match = j
        
        lut[i] = best_match
    
    # Step 4: 应用映射
    result = source_img.copy()
    for y in range(h_s):
        for x in range(w_s):
            result[y, x] = lut[source_img[y, x]]
    
    return result

In [ ]:
# ==================== 实践3: 直方图匹配 ====================

# 读取源图和参考图
source_img = cv2.imread("hist_source.jpg", cv2.IMREAD_GRAYSCALE)
reference_img = cv2.imread("hist_reference.jpg", cv2.IMREAD_GRAYSCALE)

if source_img is not None and reference_img is not None:
    print(f"读取成功!")
    print(f"源图: {source_img.shape[1]}x{source_img.shape[0]}")
    print(f"参考图: {reference_img.shape[1]}x{reference_img.shape[0]}")
    
    # 执行直方图匹配
    print("\n正在进行直方图匹配...")
    matched = histogram_matching_manual(source_img, reference_img)
    
    # 保存结果
    cv2.imwrite("histogram_matched.jpg", matched)
    print("匹配完成! 已保存: histogram_matched.jpg")
    
    # 计算各图的直方图
    hist_source = compute_histogram_manual(source_img)
    hist_ref = compute_histogram_manual(reference_img)
    hist_matched = compute_histogram_manual(matched)
    
    # 可视化对比
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 图像对比
    axes[0, 0].imshow(source_img, cmap='gray')
    axes[0, 0].set_title('Source')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(reference_img, cmap='gray')
    axes[0, 1].set_title('Reference')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(matched, cmap='gray')
    axes[0, 2].set_title('Matched Result')
    axes[0, 2].axis('off')
    
    # 直方图对比
    axes[1, 0].bar(range(256), hist_source, width=1, color='steelblue')
    axes[1, 0].set_title('Source Histogram')
    axes[1, 0].set_xlim(0, 255)
    
    axes[1, 1].bar(range(256), hist_ref, width=1, color='green')
    axes[1, 1].set_title('Reference Histogram')
    axes[1, 1].set_xlim(0, 255)
    
    axes[1, 2].bar(range(256), hist_matched, width=1, color='coral')
    axes[1, 2].set_title('Matched Histogram')
    axes[1, 2].set_xlim(0, 255)
    
    plt.tight_layout()
    plt.show()
    
    # 统计信息
    print(f"\n源图均值: {source_img.mean():.1f}")
    print(f"参考图均值: {reference_img.mean():.1f}")
    print(f"匹配结果均值: {matched.mean():.1f}")
    print("\n说明: 匹配结果的直方图应该与参考图相似")
else:
    print("读取图像失败!")

## 四、本章总结

### 核心知识点
1. **直方图计算**：统计每个灰度值出现的像素数
2. **直方图均衡化**：通过 CDF 映射使直方图均匀分布
3. **直方图匹配**：将源图直方图转换为参考图的分布

### 关键步骤
```
直方图计算: 遍历像素 → 统计频数 → 生成 hist[256]
均衡化:  直方图 → CDF → 映射表 LUT → 应用映射
匹配:    源直方图CDF + 参考直方图CDF → 最近邻匹配 → 映射
```

### 公式
- CDF: `CDF[i] = Σ(hist[0..i]) / total_pixels`
- 均衡化映射: `LUT[i] = int(CDF[i] × 255)`
- 匹配映射: `LUT[i] = argmin_j |ref_CDF[j] - src_CDF[i]|`

### 注意事项
1. 直方图是对整张图的统计，丢失空间信息
2. 均衡化对暗图效果明显，对本身已均匀的图效果有限
3. 匹配效果取决于源图和参考图的相似程度
4. 处理彩色图像时应在各通道独立进行
